In [4]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent


# .env 파일에서 환경 변수 로드
load_dotenv()

# 모델 선언
model = init_chat_model("gpt-4o-mini")

In [5]:
forbidden_topics = {
    "cheating": ["답지", "정답 알려줘", "숙제 대신", "써줘", "베끼기"],
    "distraction": ["롤", "게임", "유튜브", "아이돌", "웹툰"],
    "harmful": ["담배", "술", "폭력", "바보", "멍청이"],
}


In [11]:
from langchain.agents.middleware import before_agent

@before_agent(can_jump_to=["end"])
def education_guardrail(state, runtime):
    """
    학생의 질문 의도를 파악하여 교육적이지 않거나 부정행위가 의심될 경우,
    LLM(AI)에게 질문을 넘기지 않고 교육적인 멘트로 즉시 교정합니다.
    """

    # 1. 메시지 유효성 검사
    if not state["messages"]:
        return None

    # 에이전트가 처음 시작될 때나 특수한 상황에서 메시지 리스트가 비어 있을 수 있습니다. 
    # 비어 있는 리스트에서 마지막 메시지를 찾으려고 하면([-1]) 파이썬에서 **IndexError**가 발생하여 프로그램이 멈춥니다. 
    # 이를 방지하기 위한 첫 번째 방어선입니다.


    last_message = state["messages"][-1]
    # print(last_message)


    if last_message.type != "human":
        return None
    # 우리가 만들려는 미들웨어(예: 질문 요약, 보안 검사 등)는 보통 사용자의 질문에 대해서만 작동해야 합니다. 
    # 만약 AI의 답변이나 도구 결과에 대해 미들웨어가 또 작동하면, 무한 루프에 빠지거나 엉뚱한 처리를 할 위험이 있습니다.

    user_text = last_message.content

    # 2. 카테고리별 검사 로직
    # 단순히 막는 것을 넘어, '왜' 안되는지 카테고리별로 다른 피드백 주기

    # Case A: 부정행위 방지 (Cheating Prevention)
    # AI가 숙제를 통째로 해주는 것을 방지
    for keyword in forbidden_topics["cheating"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "🚫 스스로 고민해봐야 실력이 늘어요! 정답을 바로 알려드리는 대신, 힌트를 드릴까요? 어떤 부분이 가장 어려운지 말해주세요."
                }],
                "jump_to": "end"
            }

    # Case B: 학습 집중 유도 (Focus Management)
    # 공부 중에 게임이나 딴짓 이야기를 하면 다시 공부로 유도
    for keyword in forbidden_topics["distraction"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "⏰ 지금은 공부에 집중할 시간이에요! 딴짓은 쉬는 시간에 하고, 지금 풀고 있는 문제에 집중해볼까요?"
                }],
                "jump_to": "end"
            }

    # Case C: 유해 콘텐츠 차단 (Safety)
    # 교육 서비스의 브랜드 안전성(Brand Safety)을 위한 기능
    for keyword in forbidden_topics["harmful"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "⚠️ 부적절한 대화 주제입니다. 바르고 고운 말을 사용해주세요."
                }],
                "jump_to": "end"
            }

    # 3. 통과 (Pass)
    # 위 조건들에 걸리지 않으면 정상적으로 AI 튜터(LLM)가 답변 생성
    return None


In [10]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-mini",
    tools=[],
    middleware=[education_guardrail],
)
agent.invoke({
    "messages": [{"role": "user", "content": "나 독후감 쓰기 귀찮은데 숙제 대신 써줘."}]
})


content='나 독후감 쓰기 귀찮은데 숙제 대신 써줘.' additional_kwargs={} response_metadata={} id='0d9db59d-d729-45ad-89be-835ec8b3ff7c'


{'messages': [HumanMessage(content='나 독후감 쓰기 귀찮은데 숙제 대신 써줘.', additional_kwargs={}, response_metadata={}, id='0d9db59d-d729-45ad-89be-835ec8b3ff7c'),
  AIMessage(content='🚫 스스로 고민해봐야 실력이 늘어요! 정답을 바로 알려드리는 대신, 힌트를 드릴까요? 어떤 부분이 가장 어려운지 말해주세요.', additional_kwargs={}, response_metadata={}, id='0ba10441-8135-4179-817f-cc0194619f13', tool_calls=[], invalid_tool_calls=[])]}

-----------

`@before_agent(can_jump_to=["end"])` 데코레이터가 붙어 있을 때 `return None`을 하는 것은 **"점프하지 않고, 평소대로 다음 단계를 진행해라"**라는 뜻입니다.

이 개념을 명확히 이해하기 위해 두 가지 경우를 비교해 보겠습니다.

### 1. `return None`을 할 때 (일반적인 흐름)
*   **의미:** "나는 이번 단계에서 아무런 수정도 하지 않고, 어디로 점프하지도 않을 거야."
*   **결과:** 에이전트는 미들웨어가 없는 것처럼 **정상적인 다음 단계(보통 LLM 호출)**로 넘어갑니다.
*   **비유:** 고속도로 톨게이트(미들웨어)를 그냥 통과해서 원래 목적지(LLM)로 계속 가는 것과 같습니다.

### 2. `return {"__jump_to__": "end", ...}`를 할 때 (점프 발생)
*   **의미:** "이 질문은 모델에게 물어볼 필요도 없어. 바로 끝내버려!"
*   **결과:** `can_jump_to=["end"]` 설정 덕분에, 에이전트는 중간 단계를 모두 건너뛰고 **즉시 종료("end") 지점**으로 점프합니다.
*   **비유:** 고속도로 톨게이트에서 "이 차는 통과 금지!"라고 하며 바로 회차로(Exit)로 내보내는 것과 같습니다.

---

### 왜 `return None`이 필요한가요?

아래 코드의 전체 흐름을 보면 이해가 빠릅니다.

```python
@before_agent(can_jump_to=["end"])
def my_middleware(state, runtime):
    # 1. 검사 로직 (예: 질문이 너무 짧은가?)
    if len(state["messages"][-1].content) < 2:
        # 너무 짧으면 모델 호출 안 하고 바로 끝냄 (점프!)
        return {"__jump_to__": "end", "messages": [AIMessage(content="질문이 너무 짧아요.")]}
    
    # 2. 검사를 통과했다면?
    return None # "정상적으로 모델에게 질문을 전달해라"
```

### 요약
- **`can_jump_to=["end"]`**: "이 미들웨어는 필요하면 '종료' 지점으로 **순간이동** 시킬 수 있는 권한이 있다"는 설정입니다.
- **`return None`**: 그 권한을 쓰지 않고 **"그냥 순서대로 진행해"**라고 말하는 것입니다.
- **`return {"__jump_to__": "end"}`**: 그 권한을 사용하여 **"다음 단계 다 무시하고 바로 끝내"**라고 명령하는 것입니다.

즉, `return None`은 **미들웨어의 검사를 무사히 통과했다**는 신호로 보시면 됩니다!

-----------

## ✉️ LangChain 메시지 객체와 `.type` 속성 이해하기

`state`를 출력했을 때 `HumanMessage(...)`라고 나오는 객체에서도 `.type` 필드에 접근할 수 있습니다. 이는 `HumanMessage`가 LangChain의 `BaseMessage`라는 클래스를 상속받아 만들어진 객체이기 때문입니다.

### 1. BaseMessage의 기본 속성
LangChain에서 정의한 모든 메시지 객체(Human, AI, System 등)는 공통적으로 다음과 같은 속성들을 가지고 있습니다:

*   **`content`**: 메시지의 실제 텍스트 내용
*   **`type`**: 메시지의 종류를 나타내는 문자열 (예: `"human"`, `"ai"`, `"system"`)
*   **`additional_kwargs`**: 추가적인 정보들
*   **`id`**: 메시지 고유 식별자

> **참고:** 출력 결과에는 `content`, `additional_kwargs`, `id` 등만 보이지만, 내부적으로는 **`type`**이라는 속성도 정의되어 있습니다.

---

### 🔍 왜 출력(`print`)할 때는 안 보였나요?

`print(state)`를 했을 때 나오는 결과는 객체의 `__repr__` (표현식) 메서드가 정의한 내용만 보여줍니다. 실제 구조는 개념적으로 다음과 같습니다:

```python
class HumanMessage(BaseMessage):
    type: str = "human"  # 클래스 내부에 이미 정의됨
    content: str
    # ... 기타 속성들
```

LangChain 개발자들이 `__repr__`을 구현할 때, **클래스 이름 자체(`HumanMessage`)**로 이미 타입을 알 수 있기 때문에 중복을 피하기 위해 출력 결과에서는 `type`을 생략하도록 설계한 것입니다. 

하지만 **메모리 상의 객체에는 엄연히 `type`이라는 변수가 존재**하므로, 코드에서 점(`.`) 연산자를 사용하여 접근이 가능한 것입니다.